# C1 lab: document loading and parsing (builds the v0 snapshot)

Lesson: [`concepts/C1_document_parsing.md`](../concepts/C1_document_parsing.md), Part H. Concept card: [02 C1](../02_RAG_MASTER_LEARNING_PLAN.md#c1). Data model: [06 §3.1 and §5.3](../06_RAG_PROJECT_AND_EVALUATION_PLAN.md#53-corpus-freezing).

**How this lab works (blank-file first).**
- Each task has a **task cell**, an **empty code cell** for you, and a **CHECK cell** (`# CHECK`, do not edit). Run the CHECK cell after your code. If it fails, read the message and fix your code.
- This notebook has **no solution code**. The CHECK cells compare fingerprints, so the expected answers stay hidden until you have predicted.
- Where a task says **Predict first**, fill the prediction cell **before** you compute anything.
- **No API calls.** Never print values from `.env`. Kernel: the project **`.venv` (Python 3.12)**, where `pymupdf` is pinned at **1.26.7**.
- **What you build here is v0's input.** Tasks 1–4 produce the pages and the frozen snapshot that C2 (chunking) and C3 (embedding) read. Tasks 5–9 are diagnosis: nothing in them changes the snapshot.

## Setup (given)
Run this cell. It names the input files, defines the fingerprint helper `_fp`, the `Page` record from 06 §3.1 (do not change its fields: C2 and C3 rely on them), and `build_zoo`, which writes a small synthetic PDF with known artifacts (a toy, for Tasks 7–8).

In [ ]:
import hashlib, json, os, re, tempfile, unicodedata
from dataclasses import dataclass
import pymupdf as fitz

BASE = r"C:\Yash\New-Work\Book RAG"
PDF = os.path.join(BASE, "upstream", "RAG_Techniques", "data", "Understanding_Climate_Change.pdf")   # Corpus D
BOOK_CH1 = os.path.join(BASE, "RAG_Made_Simple_The_Complete_Visual_Guide", "chapter 1.pdf")         # specimen only, not a corpus
SNAP_DIR = os.environ.get("C1_SNAP_DIR", os.path.join(BASE, "rag_learning", "evaluation", "snapshots"))  # 06 §5.3
ZOO_PDF = os.path.join(tempfile.gettempdir(), "c1_zoo.pdf")                                            # outside the workspace

def _fp(x):
    """Fingerprint of a value. The CHECK cells compare fingerprints, not visible answers."""
    return hashlib.sha256(json.dumps(x, sort_keys=True).encode()).hexdigest()[:16]

def sha256_hex(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

@dataclass(frozen=True)
class Page:                 # 06 §3.1: Page(doc_id, page=1-based, text, parser="pymupdf <version>")
    doc_id: str
    page: int               # 1-based PDF page
    text: str               # raw page.get_text(); v0 does no cleaning
    parser: str             # e.g. "pymupdf 1.26.7"

# ---- the artifact zoo (a toy PDF; copied verbatim from sources/verification/C1/c1_zoo.py) ----
import pymupdf as fitz

ZOO_HEADER = "ACME Technical Report"
ZOO_LEFT = ["Parsing comes before chunking:", "a bad parse cannot be fixed", "by any later extrac-", "tion or retrieval step."]
ZOO_RIGHT = ["Two-column pages are common", "in research papers, and the", "reading order must follow", "each column top to bottom."]
ZOO_LIG = "The \ufb01rst \ufb02ag: check every \ufb01le."
ZOO_TABLE = [["Model", "Max tokens"], ["MiniLM-L6", "256"], ["bge-small", "512"]]
ZOO_SCAN = "This sentence exists only as pixels."
ZOO_P5 = ["Page five checks the text extraction", "and repeats the header and footer."]


def build_zoo(path):
    doc = fitz.open()
    lig_font = fitz.Font("cjk")            # a built-in font that has the ligature glyphs

    def frame(page, n):
        page.insert_text((72, 50), ZOO_HEADER, fontname="helv", fontsize=9)
        page.insert_text((72, 800), f"Page {n}", fontname="helv", fontsize=9)

    p = doc.new_page(width=595, height=842)          # page 1
    frame(p, 1)
    y = 100
    for a, b in zip(ZOO_LEFT, ZOO_RIGHT):            # line by line ACROSS the columns
        p.insert_text((72, y), a, fontname="helv", fontsize=10)
        p.insert_text((320, y), b, fontname="helv", fontsize=10)
        y += 14
    tw = fitz.TextWriter(p.rect)
    tw.append((72, 200), ZOO_LIG, font=lig_font, fontsize=10)
    tw.write_text(p)

    p = doc.new_page(width=595, height=842)          # page 2: ruled table
    frame(p, 2)
    x0, x1, x2, y0, h = 72, 222, 372, 100, 20
    for r, row in enumerate(ZOO_TABLE):
        top = y0 + r * h
        p.draw_rect(fitz.Rect(x0, top, x1, top + h), color=(0, 0, 0), width=0.8)
        p.draw_rect(fitz.Rect(x1, top, x2, top + h), color=(0, 0, 0), width=0.8)
        p.insert_text((x0 + 4, top + 14), row[0], fontname="helv", fontsize=10)
        p.insert_text((x1 + 4, top + 14), row[1], fontname="helv", fontsize=10)

    doc.new_page(width=595, height=842)              # page 3: empty

    src = fitz.open()                                # page 4: an image of text, no text layer
    sp = src.new_page(width=400, height=60)
    sp.insert_text((10, 35), ZOO_SCAN, fontname="helv", fontsize=14)
    pix = sp.get_pixmap(dpi=100)
    p = doc.new_page(width=595, height=842)
    p.insert_image(fitz.Rect(72, 100, 472, 160), pixmap=pix)

    p = doc.new_page(width=595, height=842)          # page 5
    frame(p, 5)
    for i, line in enumerate(ZOO_P5):
        p.insert_text((72, 100 + 14 * i), line, fontname="helv", fontsize=10)
    doc.save(path)
    return path

build_zoo(ZOO_PDF)
print("pymupdf", fitz.VersionBind, "| snapshot folder:", SNAP_DIR, "| ready")

## Task 1: load the PDF page by page

**Predict first** (cell below): how many pages does Corpus D have, and what number will **your** first page carry?

Then write `load_pages(pdf_path, doc_id)` that returns a list of `Page`, one per PDF page, in order:
- `page` is **1-based** (the first page is 1);
- `text` is exactly `page.get_text()`: no strip, no cleaning;
- `parser` is `f"pymupdf {fitz.VersionBind}"`.

Then run `pages = load_pages(PDF, "climate")`.

In [ ]:
# PREDICTION
my_prediction_n_pages = None        # an integer
my_prediction_first_page = None     # 0 or 1

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from c1_ref.py or c1_measure.py).

In [ ]:
# CHECK Task 1 (do not edit)
assert my_prediction_n_pages is not None and my_prediction_first_page is not None, "fill the prediction cell first"
assert fitz.VersionBind == "1.26.7", f"pymupdf must be the pinned 1.26.7, got {fitz.VersionBind}: snapshot_id depends on it"
assert isinstance(pages, list) and all(isinstance(p, Page) for p in pages), "pages must be a list of Page"
assert [p.page for p in pages] == list(range(1, len(pages) + 1)), "page numbers must be 1, 2, 3, ... (1-based, in order)"
assert {p.doc_id for p in pages} == {"climate"} and {p.parser for p in pages} == {"pymupdf 1.26.7"}, "doc_id / parser fields"
assert _fp([len(p.text) for p in pages]) == "b2aee2f144d78306", "page texts differ from raw page.get_text(): did you strip or clean?"
print("Task 1 OK:", len(pages), "pages; you predicted", my_prediction_n_pages, "| shortest page:",
      min(pages, key=lambda p: len(p.text)).page, "with", min(len(p.text) for p in pages), "characters")

## Task 2: one document text, one page table

v0 has **one coordinate system**: offsets into `doc_text`, the pages joined with `"\n"` (06 §2.3).

**Predict first:** the 33 page texts add up to `sum(len(p.text) for p in pages)` characters (print that number first). How long will `doc_text` be?

Write `build_page_table(page_texts)` that returns `(doc_text, rows)`, where each row is a dict with exactly these keys (06 §5.3):
`{"page": 1-based, "doc_char_start": ..., "doc_char_end": ..., "text_sha256": sha256_hex(page text)}`.
Rule: `doc_text[row["doc_char_start"]:row["doc_char_end"]]` must equal that page's text.

In [ ]:
# PREDICTION
my_prediction_len_doc_text = None   # an integer, BEFORE you build doc_text

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from c1_ref.py or c1_measure.py).

In [ ]:
# CHECK Task 2 (do not edit)
assert my_prediction_len_doc_text is not None, "fill the prediction cell first"
doc_text, rows = build_page_table([p.text for p in pages])
assert all(doc_text[r["doc_char_start"]:r["doc_char_end"]] == p.text for r, p in zip(rows, pages)), "a slice does not give back its page"
assert _fp(len(doc_text)) == "b3a90a6e1e17104b", "len(doc_text) is not right yet: how many join characters are there?"
assert _fp(rows) == "3142634c0e4e06a8", "rows differ: check the keys, 1-based page numbers, and the +1 after each page"
print("Task 2 OK. len(doc_text) =", len(doc_text), "| you predicted", my_prediction_len_doc_text,
      "| page 8 starts at", rows[7]["doc_char_start"], "(06 §5.3 says 16,074)")

## Task 3: the `snapshot_id`

Write `snapshot_id(backend, version, options, cleaning_version, doc_text_sha256)` by copying the **exact formula in 06 §5.3**. `doc_text_sha256` is a dict `{doc_id: sha256_hex(doc_text)}`. Watch the comment about `"page_join=\\n"`.

Then compute `sid = snapshot_id("pymupdf", fitz.VersionBind, {}, "none", {"climate": sha256_hex(doc_text)})`.

**Predict first:** suppose a newer pymupdf (say 1.28.2) extracted **byte-identical** text. Would the id stay the same?

In [ ]:
# PREDICTION
my_prediction_same_id_under_1_28_2 = None   # True or False

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from c1_ref.py or c1_measure.py).

In [ ]:
# CHECK Task 3 (do not edit)
assert my_prediction_same_id_under_1_28_2 is not None, "fill the prediction cell first"
assert _fp(sid) == "7e681e4a5012ca59", "sid is not the 06 §5.3 value yet. Is page_join a literal backslash + n? Is options json.dumps(..., sort_keys=True)?"
h = {"climate": sha256_hex(doc_text)}
other = snapshot_id("pymupdf", "1.28.2", {}, "none", h)
assert _fp(other) == "ce1948f958b90905", "your function does not put the version into the id the 06 way"
tabs = snapshot_id("pymupdf", "1.26.7", {}, "tabs_v1", h)
assert _fp(tabs) == "eab4643a827491ac", "your function does not put the cleaning version into the id the 06 way"
print("Task 3 OK. snapshot_id =", sid, "| same text, version 1.28.2 ->", other, "| cleaning 'tabs_v1' ->", tabs)
print("You predicted 'same id under 1.28.2':", my_prediction_same_id_under_1_28_2)

**One sentence.** Corpus D has no tab characters at all. Why does a no-op cleaning step (`tabs_v1`) still change the id, and why is that the right design?

*Template:* "The id changes because ____ is part of the recipe. This is right because ____."

*Your answer:*

## Task 4: write the snapshot, then refuse a broken one

1. Build `manifest`, a dict with the **06 §5.3 fields**: `snapshot_id, corpus ("D"), corpus_version ("1.0"), created_utc, parser, cleaning, encoding, newline, page_join, docs`. `parser` = `{"backend": "pymupdf", "version": fitz.VersionBind, "call": "page.get_text()", "options": {}}`; `cleaning` = `{"version": "none", "steps": []}`; `docs` = one dict with `doc_id, pdf_sha256, bytes, pages, doc_text_sha256, page_table` (the page table is your `rows`).
2. Write `doc_text` to `SNAP_DIR/<sid>/climate.txt` and the manifest to `SNAP_DIR/D_<sid>.json` (`json.dumps(manifest, indent=2, ensure_ascii=False)`), both UTF-8. Set `text_path` and `manifest_path` to the two paths.
3. Write `validate_snapshot(manifest, doc_texts)` (`doc_texts = {doc_id: text}`) that **raises `ValueError`** unless: the text hash matches; pages are 1..n starting at 0; each start = previous end + 1; the last end = `len(text)`; each page slice has its `text_sha256`; and the recomputed `snapshot_id` matches.

**Predict first:** you are on Windows. If you write the text with a plain `open(path, "w", encoding="utf-8")`, will the file's sha256 equal `doc_text_sha256`?

In [ ]:
# PREDICTION
my_prediction_plain_open_hash_matches = None   # True or False

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from c1_ref.py or c1_measure.py).

In [ ]:
# CHECK Task 4 (do not edit)
import copy
assert my_prediction_plain_open_hash_matches is not None, "fill the prediction cell first"
need = {"snapshot_id", "corpus", "corpus_version", "created_utc", "parser", "cleaning", "encoding", "newline", "page_join", "docs"}
assert set(manifest) == need, f"manifest keys differ from 06 §5.3: {sorted(set(manifest) ^ need)}"
d0 = manifest["docs"][0]
assert set(d0) == {"doc_id", "pdf_sha256", "bytes", "pages", "doc_text_sha256", "page_table"}, "docs[0] keys differ from 06 §5.3"
assert _fp([d0["pdf_sha256"], d0["bytes"]]) == "3bf65abc92c1f427", "pdf_sha256 / bytes: hash the PDF file's bytes"
assert manifest["snapshot_id"] == sid and d0["page_table"] == rows and manifest["page_join"] == "\n"
raw = open(text_path, "rb").read()
assert hashlib.sha256(raw).hexdigest() == d0["doc_text_sha256"], \
    f"the file's bytes do not hash to doc_text_sha256 ({len(raw)} bytes on disk vs {len(doc_text.encode())}). Hint: newline="
assert json.load(open(manifest_path, encoding="utf-8"))["snapshot_id"] == sid
validate_snapshot(manifest, {"climate": doc_text})                      # must pass
bad = copy.deepcopy(manifest)
bad["docs"][0]["page_table"][1]["doc_char_start"] += 1                 # page 2 shifted by one,
bad["docs"][0]["page_table"][1]["doc_char_end"] += 1                   # same length, same hash
try:
    validate_snapshot(bad, {"climate": doc_text})
    raise AssertionError("validate_snapshot accepted a page table with page 2 shifted by +1: it must fail closed")
except ValueError as e:
    print("shifted page table rejected:", e)
print("Task 4 OK. Wrote", text_path, "and", manifest_path)

## Task 5: from a document offset back to a page

Citations need a page. Write `page_of(doc_offset, rows)` that returns `(page, page_offset)`: the 1-based page that contains `doc_offset`, and the offset inside that page's text. Rule (07 H.3): the joining `"\n"` after page *p* belongs to page *p*.

Then set `paris = page_of(doc_text.find("Paris Agreement"), rows)` and `paris_pages`, the sorted list of **distinct** pages on which "Paris Agreement" occurs.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from c1_ref.py or c1_measure.py).

In [ ]:
# CHECK Task 5 (do not edit)
offs = [0, 2423, 2424, 2425, 16074, len(doc_text) - 1]
got = [list(page_of(o, rows)) for o in offs]
assert _fp(got) == "718b2430f71084f0", f"page_of is not right yet: {got}. Which page owns offset 2424 (the join after page 1)?"
assert _fp([list(paris), list(paris_pages)]) == "b10ac9a73363481c", "paris / paris_pages are not right yet"
print("Task 5 OK. First 'Paris Agreement' is on page", paris[0], "at page offset", paris[1], "| pages:", paris_pages)

## Task 6: an artifact inventory of Corpus D (raw text, nothing changed)

Write `artifact_counts(page_texts)` that returns a dict with:
- `"empty_pages"`: 1-based pages whose text is only whitespace;
- `"hyphen_breaks"`: a list of `[page, match]` for every match of `re.compile(r"([A-Za-z]+)-\n([a-z]+)")`;
- `"ligatures"`: count of the characters U+FB00 to U+FB06; `"tabs"`: count of `"\t"`; `"double_newlines"`: count of `"\n\n"`;
- `"lines"`: number of newline-terminated lines; `"lines_ending_space"`: how many of them end with `" "` before the `"\n"`.

Write `repeated_lines(page_texts)` that returns `[threshold, {key: n_pages}]`: a line's key is the stripped line with each run of digits replaced by `"#"`; count the **pages** a key appears on; `threshold = max(3, ceil(n_text_pages / 2))`; keep keys at or over it.

**Predict first:** how many of the line-end hyphens in Corpus D are real compound words (like "well-being") rather than words split by the layout?

In [ ]:
# PREDICTION
my_prediction_real_compounds = None   # e.g. "all", "none", "about half"

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from c1_ref.py or c1_measure.py).

In [ ]:
# CHECK Task 6 (do not edit)
assert my_prediction_real_compounds is not None, "fill the prediction cell first"
texts = [p.text for p in pages]
a = artifact_counts(texts)
keys = ("empty_pages", "ligatures", "tabs", "double_newlines", "lines", "lines_ending_space")
assert _fp({k: a[k] for k in keys}) == "78b0370b2f60b6ab", f"some count is not right yet: { {k: a[k] for k in keys} }"
assert _fp([list(x) for x in a["hyphen_breaks"]]) == "eded03f80e87719a", "hyphen_breaks is not right yet"
assert _fp(list(repeated_lines(texts))) == "960329243dde697d", "repeated_lines is not right yet"
for pg, m in a["hyphen_breaks"]:
    print(f"  p.{pg}: {m!r}")
print("Task 6 OK.", {k: a[k] for k in keys}, "| repeated lines:", repeated_lines(texts))

**Two sentences.** (1) Would a rule "delete every `-\n`" help or hurt Corpus D? (2) C2's splitter tries `"\n\n"` first. What does your `double_newlines` count mean for it?

*Template:* "(1) It would ____ because ____. (2) The `\n\n` separator ____ on this text, so the splitter ____."

*Your answer:*

## Task 7: diagnose the artifact zoo (toy PDF)

The zoo (`ZOO_PDF`, 5 pages) has a running header and footer, two columns on page 1 written line by line across the page, a word hyphenated at a line end, three ligature glyphs, a ruled table, an empty page and an image-only ("scanned") page.

**Predict first:** on page 1, after the header lines, which line will `get_text()` give right after "Parsing comes before chunking:"?

Then compute:
- `zoo_default`: the list of `page.get_text()` for the 5 pages, and `p1_lines = zoo_default[0].split("\n")`;
- `extract_columns(page)`: the left half of the page (full height) then the right half, using `page.get_text(clip=...)`;
- `zoo_tables`: `len(page.find_tables().tables)` for each page;
- `zoo_kinds`: for each page `"text"` (non-whitespace text), else `"image_only"` (it has images), else `"empty"`;
- `lig_count`: ligature characters (U+FB00–U+FB06) in `extract_columns(page 1)`.

In [ ]:
# PREDICTION
my_prediction_line_after_parsing = None   # the exact line you expect

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from c1_ref.py or c1_measure.py).

In [ ]:
# CHECK Task 7 (do not edit)
assert my_prediction_line_after_parsing is not None, "fill the prediction cell first"
assert _fp(p1_lines) == "98f249bd90aed81b", "p1_lines must be the raw get_text() of page 1 split on newlines"
with fitz.open(ZOO_PDF) as _d:
    _col = extract_columns(_d[0])
assert _fp(_col) == "edbd2f4ad8b1b110", "extract_columns: left half (full height) then right half, plain get_text(clip=...)"
assert _fp(zoo_tables) == "258f794e4d5d8820", f"zoo_tables is not right yet: {zoo_tables}"
assert _fp(zoo_kinds) == "ed8fa3bfa5db7bff", f"zoo_kinds is not right yet: {zoo_kinds}"
assert _fp(lig_count) == "4e07408562bedb8b", "lig_count is not right yet"
i = p1_lines.index("Parsing comes before chunking:")
print("Task 7 OK. After 'Parsing comes before chunking:' get_text() gives:", repr(p1_lines[i + 1]))
print("You predicted:", repr(my_prediction_line_after_parsing), "| tables per page:", zoo_tables, "| kinds:", zoo_kinds)
print("'file' found in page 1 text:", "file" in _col, "| after NFKC:", "file" in unicodedata.normalize("NFKC", _col))

## Task 8: cleaning rules that log every change (analysis only; v0 applies none)

An **edit** is a dict `{"doc_id", "page", "rule", "start", "end", "before", "after"}` with **page-relative** raw offsets: it says "raw `text[start:end]` (= `before`) becomes `after`".

Write `plan_edits(doc_id, page_texts)` with three rules, edits sorted by `(page, start)`:
- **R1_header_footer:** every line (including its `"\n"`) whose key (Task 6) is in `repeated_lines(page_texts)`; `after = ""`.
- **R2_dehyphenate:** for each match of the Task 6 regex, if `(left + right).lower()` occurs as a word (`[A-Za-z]+`, lowercased) anywhere in `page_texts`, delete **only** the `"-\n"`: `start` = end of the left fragment, `end = start + 2`, `before = "-\n"`, `after = ""`.
- **R3_ligature:** each character U+FB00–U+FB06 becomes `unicodedata.normalize("NFKC", ch)`.

Write `apply_edits(page_text, edits) -> (cleaned, src)` where `src[i]` is the raw offset of cleaned character `i` (an inserted character maps to its edit's `start`), and `undo_edits(cleaned, src, edits, raw_len) -> raw` that rebuilds the raw text.

Use `zoo_pages = [extract_columns(page 1)] + zoo_default[1:]` and set `zoo_edits = plan_edits("zoo", zoo_pages)`.

**Predict first:** how many edits will the same three rules make on Corpus D?

In [ ]:
# PREDICTION
my_prediction_edits_on_D = None   # an integer

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from c1_ref.py or c1_measure.py).

In [ ]:
# CHECK Task 8 (do not edit)
assert my_prediction_edits_on_D is not None, "fill the prediction cell first"
got = [[e["page"], e["rule"], e["start"], e["end"], e["before"], e["after"]] for e in zoo_edits]
assert _fp(got) == "da5d0c477903b863", "zoo_edits are not right yet: compare each rule with the spec (R2 deletes only '-\\n')"
for i, t in enumerate(zoo_pages, 1):
    pe = [e for e in zoo_edits if e["page"] == i]
    cleaned, src = apply_edits(t, pe)
    assert len(src) == len(cleaned), "src needs one raw offset per cleaned character"
    assert undo_edits(cleaned, src, pe, len(t)) == t, f"page {i}: undo(apply(edits)) does not give back the raw text"
d_edits = plan_edits("climate", [p.text for p in pages])
naive = sum(len(re.findall(r"([A-Za-z]+)-\n([a-z]+)", p.text)) for p in pages)
assert _fp([len(d_edits), naive]) == "9256b3c60b4c2079", "plan_edits on Corpus D is not right yet"
by_rule = {}
for e in zoo_edits:
    by_rule[e["rule"]] = by_rule.get(e["rule"], 0) + 1
print("Task 8 OK. Zoo edits by rule:", by_rule, "| Corpus D edits:", len(d_edits), "(you predicted", my_prediction_edits_on_D, ")",
      "| a naive '-\\n' join would change", naive)
c1, s1 = apply_edits(zoo_pages[0], [e for e in zoo_edits if e["page"] == 1])
k = c1.find("extraction")
print("cleaned 'extraction' comes from raw page-1 offsets", s1[k], "to", s1[k + 9], ":", repr(zoo_pages[0][s1[k]:s1[k + 9] + 1]))

## Task 9: a real page with a figure and a table (the book, as a specimen)

Open `BOOK_CH1` (read-only). On **page 2**: count the images, and count the text blocks (`page.get_text("blocks")`) whose rectangle intersects the image's rectangle (`page.get_image_rects(xref)`). On **page 13**: count tables with `find_tables()` and with `find_tables(strategy="text")`.

Set `book_obs = [images_p2, blocks_inside_image_p2, tables_p13_default, tables_p13_text]`. Then look at the page images yourself (open the PDF) and compare with `get_text()`.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from c1_ref.py or c1_measure.py).

In [ ]:
# CHECK Task 9 (do not edit)
assert _fp(list(book_obs)) == "3743df1303f54c6c", f"book_obs is not right yet: {book_obs}"
print("Task 9 OK.", book_obs)

**One sentence each.** (1) What does `get_text()` give you from the figure on page 2? (2) What happened to the table's rows on page 13?

*Template:* "(1) From the figure I get ____. (2) The rows became ____, so a chunk could ____."

*Your answer:*

## Hand-off to C2 and C3 (run, do not edit)

This prints the exact shape that the next lessons read. Copy the printed `snapshot_id` into your notes.

In [ ]:
print("pages      :", type(pages).__name__, "of", type(pages[0]).__name__, "| n =", len(pages),
      "| fields:", list(Page.__dataclass_fields__))
print("doc_text   : str, len", len(doc_text), "| sha256", sha256_hex(doc_text)[:16], "| pages joined with '\\n'")
print("rows       : page_table, keys", list(rows[0]))
print("snapshot_id:", manifest["snapshot_id"], "| cleaning:", manifest["cleaning"]["version"])
print("files      :", text_path, "|", manifest_path)

## Wrap-up: evidence for the log

Answer each in one line, no notes.

1. *Decision:* "v0 applies ____ cleaning, because offsets must refer to ____. A cleaning rule would create a new ____."
2. *Two extractors:* "The repo's `encode_pdf` and `read_pdf_to_string` give different ____, so the same character has ____."
3. *Corpus T (pending the file list, D-01/D-02):* "When T is chosen, I will rerun Tasks 1–8 on it and log every artifact with ____ and ____."

> Tell the tutor when you are done. The tutor grades `coding_independence_0_3` (Tasks 1, 2, 3, 5) and `debugging_0_3` (any failed CHECK you fixed yourself, and Task 4).